# Banana-slug IDBac application
Derived from the preserved genome-workup notebook, cells 1 and 28–32.
Run from this notebook directory. Inputs are frozen tables; no genome analysis or remote search is needed.
The original notebook and figures are preserved alongside this notebook. Regenerated figures go to `regenerated_figures/`.
The six scoring labels are the normalized GTDB genera used in the original analysis; 16S and QC tables are also preserved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
import seaborn as sns

DATA = Path("../data")
FIGS = Path("regenerated_figures")
FIGS.mkdir(exist_ok=True)

# Presentation-friendly defaults (Arial with fallbacks for non-Windows systems)
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "mathtext.default": "regular",
})
import warnings
warnings.filterwarnings("ignore", message="findfont")
sns.set_palette("deep")

# Consistent colors
C_PASS = "#2a9d8f"
C_WARN = "#e9c46a"
C_FAIL = "#e76f51"
C_OTHER = "#bdbdbd"


In [ ]:
# Fig 6.1b — real IDBac hit data from enriched_db_results.tsv
idbac = pd.read_csv(DATA / "idbac/enriched_db_results.tsv", sep="\t")
idbac["sl"] = idbac["query_filename"].str.extract(r"(SL\d+)")
hits = idbac[idbac["distance"] <= 0.232].copy()

# The 6 strains that returned hits in the banana-slug cohort, in the order used for the figure.
# True genus taken from GTDB-tk (normalized).
STRAINS_6 = [
    ("SL1003", "Mucilaginibacter"),  # correct top-1
    ("SL1097", "Pseudomonas"),       # correct top-1
    ("SL1102", "Pseudomonas"),       # correct top-1
    ("SL1105", "Pseudomonas"),       # correct top-1
    ("SL1033", "Pseudomonas"),       # WRONG top-1 (Priestia), rescued by density
    ("SL1072", "Pseudomonas"),       # WRONG top-1 (Acinetobacter), rescued by density
]

# Rank hits per strain by ascending distance; mark each "correct" if hit genus == true genus.
per_strain = {}
for sl, true_genus in STRAINS_6:
    sub = (hits[hits["sl"] == sl]
           .sort_values("distance")
           .reset_index(drop=True))
    per_strain[sl] = [int(g == true_genus) for g in sub["db_genus"]]
    print("{:>6}  true={:<18}  n_hits={}  correct={}/{}".format(
        sl, true_genus, len(sub), sum(per_strain[sl]), len(sub)))

# ---- Plot ----
fig, ax = plt.subplots(figsize=(13, 4.6))

max_len = max(len(h) for h in per_strain.values())
tile_h = 0.78
tile_w = 0.68   # more space between tiles (gap ≈ 0.32)

for i, (sl, true_genus) in enumerate(STRAINS_6):
    hit_row = per_strain[sl]
    y = len(STRAINS_6) - 1 - i
    for j, h in enumerate(hit_row):
        color = C_PASS if h == 1 else C_FAIL
        ax.add_patch(plt.Rectangle((j, y - tile_h/2), tile_w, tile_h,
                                   facecolor=color, edgecolor="black", linewidth=0.3))
    # Thicker border on the top-1 tile
    ax.add_patch(plt.Rectangle((0, y - tile_h/2), tile_w, tile_h,
                               facecolor="none", edgecolor="black", linewidth=2.0))
    ax.text(-0.6, y, "{} · {}".format(sl, true_genus),
            ha="right", va="center", fontsize=10)

ax.set_xlim(-12, max_len + 1)
ax.set_ylim(-0.7, len(STRAINS_6))
ax.set_yticks([])
# X-ticks every 10 hits (rank 1, 10, 20, ...)
tick_positions = [0] + list(range(9, max_len, 10))
tick_labels    = [str(p + 1) for p in tick_positions]
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)
ax.set_xlabel("Hit Rank")
ax.set_title("Fig 6.1b — Density of correct genus-level hits rescues top-1 errors")
for s in ("top", "left", "right"):
    ax.spines[s].set_visible(False)

handles = [
    mpatches.Patch(color=C_PASS, label="correct genus"),
    mpatches.Patch(color=C_FAIL, label="incorrect genus"),
    mpatches.Patch(facecolor="white", edgecolor="black", linewidth=2.0, label="top-1"),
]
ax.legend(handles=handles, loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig6_1b_density.png")
plt.show()


In [ ]:
# Paper-figure candidates for Fig 6.1b. Saved to notebooks/figures/fig6_iterations/
# so we can iterate without overwriting earlier versions.
FIG_ITER = FIGS / "fig6_iterations"
FIG_ITER.mkdir(exist_ok=True)

# Recompute per-strain ranked hit genera and correctness
STRAINS_6 = [
    ("SL1033", "Pseudomonas"),    # rescue case — put first for paper
    ("SL1072", "Pseudomonas"),    # rescue case
    ("SL1003", "Mucilaginibacter"),
    ("SL1097", "Pseudomonas"),
    ("SL1102", "Pseudomonas"),
    ("SL1105", "Pseudomonas"),
]

def strain_hits(sl):
    return (hits[hits["sl"] == sl].sort_values("distance")
            .reset_index(drop=True)["db_genus"].tolist())

per_strain = {sl: strain_hits(sl) for sl, _ in STRAINS_6}
per_strain_correct = {sl: [int(g == true_g) for g in per_strain[sl]]
                      for sl, true_g in STRAINS_6}
per_strain_precision = {sl: (sum(hits_bool) / len(hits_bool) if hits_bool else 0.0)
                        for sl, hits_bool in per_strain_correct.items()}


In [ ]:
# Variant A — tiles + right-hand precision gauge
# Rendered twice: top-10 (for main text) and all hits (for SI).
def plot_tiles_with_gauge(limit, fname, title, fig_w, show_title=True, legend_below=False):
    tile_h = 0.80
    tile_w = 0.70            # generous gap between tiles
    gauge_gap = 2.5           # gap between tile area and gauge
    gauge_w = 8.0             # gauge axis width in data units
    # x extent of gauge = [gauge_x0, gauge_x0 + gauge_w]
    gauge_x0 = limit + gauge_gap
    gauge_x1 = gauge_x0 + gauge_w

    fig, ax = plt.subplots(figsize=(fig_w, 3.4))
    for i, (sl, true_g) in enumerate(STRAINS_6):
        y = len(STRAINS_6) - 1 - i
        corr = per_strain_correct[sl]
        show = corr[:limit]
        # Tiles
        for j, h in enumerate(show):
            color = C_PASS if h == 1 else C_FAIL
            ax.add_patch(plt.Rectangle((j, y - tile_h/2), tile_w, tile_h,
                                       facecolor=color, edgecolor="black", linewidth=0.3))
        # Thick border on top-1 tile
        ax.add_patch(plt.Rectangle((0, y - tile_h/2), tile_w, tile_h,
                                   facecolor="none", edgecolor="black", linewidth=1.8))

        # Row label — italicize the genus only
        ax.text(-0.6, y, "{} · $\mathit{{{}}}$".format(sl, true_g),
                ha="right", va="center", fontsize=10)

        # Precision gauge (horizontal bar from gauge_x0 to gauge_x0 + gauge_w * precision)
        prec = per_strain_precision[sl]
        ax.add_patch(plt.Rectangle((gauge_x0, y - tile_h/2 * 0.75),
                                   gauge_w, tile_h * 0.75,
                                   facecolor="#eeeeee", edgecolor="#bbb", linewidth=0.6))
        ax.add_patch(plt.Rectangle((gauge_x0, y - tile_h/2 * 0.75),
                                   gauge_w * prec, tile_h * 0.75,
                                   facecolor="#264653", edgecolor="black", linewidth=0.4))

    # Axis cosmetics
    ax.set_ylim(-0.7, len(STRAINS_6))
    ax.set_xlim(-12, gauge_x1 + 1)
    ax.set_yticks([])

    # X ticks — two groups: hit rank on the left, 0-100% for gauge on the right
    rank_ticks = [0] + list(range(9, limit, 10))
    rank_labels = [str(p + 1) for p in rank_ticks]
    # gauge ticks
    gauge_ticks  = [gauge_x0, gauge_x0 + gauge_w/2, gauge_x0 + gauge_w]
    gauge_labels = ["0%", "50%", "100%"]

    ax.set_xticks(rank_ticks + gauge_ticks)
    ax.set_xticklabels(rank_labels + gauge_labels)

    # Split x-axis labels: below the tiles say "Hit rank", below the gauges say "Precision"
    ax.text(limit / 2 - 0.5, -2.0, "Hit rank",
            ha="center", va="top", fontsize=10)
    ax.text(gauge_x0 + gauge_w / 2, -2.0, "Genus-level precision",
            ha="center", va="top", fontsize=10)

    if show_title:
        ax.set_title(title, fontsize=12)
    for s in ("top", "left", "right"):
        ax.spines[s].set_visible(False)
    # Shorten the visible bottom spine to avoid it running past the gauges
    ax.spines["bottom"].set_bounds(-0.4, gauge_x1)

    # Legend
    handles = [
        mpatches.Patch(color=C_PASS, label="Correct Genus"),
        mpatches.Patch(color=C_FAIL, label="Incorrect Genus"),
        mpatches.Patch(facecolor="white", edgecolor="black", linewidth=1.8, label="Top-1"),
        mpatches.Patch(facecolor="#264653", edgecolor="black", linewidth=0.4,
                       label="Precision (# Correct / # Hits)"),
    ]
    if legend_below:
        ax.legend(handles=handles, loc="upper center",
                  bbox_to_anchor=(0.5, -0.40), ncol=2, fontsize=9, frameon=False)
    else:
        ax.legend(handles=handles, loc="upper right",
                  bbox_to_anchor=(1.0, 1.15), ncol=4, fontsize=9)

    plt.tight_layout()
    plt.savefig(FIG_ITER / fname)
    plt.show()

# Main-text candidate: top-10 view.
# fig_w chosen so inch-per-data-unit matches the all-hits version
# (16 in / 94.5 x-units ≈ 0.169 in/unit; top-10 xlim spans 33.5 units → ~5.7 in)
plot_tiles_with_gauge(limit=10,
                       fname="tiles_top10_with_precision_gauge.png",
                       title="",
                       fig_w=5.7,
                       show_title=False,
                       legend_below=True)

# SI candidate: all hits
max_hits = max(len(per_strain_correct[sl]) for sl, _ in STRAINS_6)
plot_tiles_with_gauge(limit=max_hits,
                       fname="tiles_all_hits_with_precision_gauge.png",
                       title="",
                       fig_w=16,
                       show_title=False)


In [ ]:
# Variant B — Precision@k line plot
# Per-strain: y = (# correct in top-k) / k, for k = 1 .. n_hits
import itertools

def plot_precision_at_k(xlim_max, fname, title, fig_w):
    fig, ax = plt.subplots(figsize=(fig_w, 4.4))
    color_cycle = plt.cm.viridis(np.linspace(0.05, 0.85, len(STRAINS_6)))

    for (sl, true_g), color in zip(STRAINS_6, color_cycle):
        corr = per_strain_correct[sl]
        n = len(corr)
        ks = np.arange(1, n + 1)
        cum = np.cumsum(corr)
        prec_at_k = cum / ks
        ax.plot(ks, prec_at_k, "-o", color=color, linewidth=1.6,
                markersize=4, label="{} · {}".format(sl, true_g))
        # Emphasize top-1 point if wrong (distinct marker)
        if corr[0] == 0:
            ax.plot(1, prec_at_k[0], marker="x", markersize=10,
                    markeredgewidth=2, color="black")

    # Reference line at 1.0
    ax.axhline(1.0, ls="--", color="#999", linewidth=0.8, alpha=0.7)

    ax.set_xlim(0.5, xlim_max + 0.5)
    ax.set_ylim(-0.02, 1.08)
    ax.set_xlabel("Hit rank k")
    ax.set_ylabel("Precision at k  (n_correct / k)")
    ax.set_title(title, fontsize=12)
    ax.legend(loc="lower right", fontsize=9, frameon=False)
    plt.tight_layout()
    plt.savefig(FIG_ITER / fname)
    plt.show()

plot_precision_at_k(xlim_max=10, fig_w=7.5,
                     fname="precision_at_k_top10.png",
                     title="Candidate — Precision @ k (k = 1..10)")
plot_precision_at_k(xlim_max=max_hits, fig_w=10,
                     fname="precision_at_k_all.png",
                     title="SI candidate — Precision @ k (all hits)")


In [ ]:
# Variant C — Threshold vs. pooled precision across the 6 banana-slug strains.
# Uses the full idbac table (unfiltered by the 0.232 default cutoff) so we can see
# how genus-level precision degrades as the distance threshold is relaxed.
sl_to_true = dict(STRAINS_6)
scope = idbac[idbac["sl"].isin(sl_to_true)].copy()
scope["true_g"] = scope["sl"].map(sl_to_true)
scope["correct"] = (scope["db_genus"] == scope["true_g"]).astype(int)
scope = scope.sort_values("distance").reset_index(drop=True)
scope["n_cum"] = np.arange(1, len(scope) + 1)
scope["k_cum"] = scope["correct"].cumsum()
scope["precision"] = scope["k_cum"] / scope["n_cum"]

DEFAULT_T = 0.232
at_default = scope[scope["distance"] <= DEFAULT_T]
if len(at_default):
    n_def = len(at_default)
    p_def = at_default["correct"].sum() / n_def
    print("At default threshold {:.3f}: n={}, precision={:.3f}".format(DEFAULT_T, n_def, p_def))

fig, ax = plt.subplots(figsize=(5.7, 3.6))
ax.step(scope["distance"], scope["precision"], where="post",
        color="#264653", linewidth=1.6, zorder=3)
ax.axvline(DEFAULT_T, ls="--", color="#b55", linewidth=1.0, zorder=4)

ax2 = ax.twinx()
ax2.step(scope["distance"], scope["n_cum"], where="post",
         color="#bbb", linewidth=1.0, alpha=0.9, zorder=2)
# Put the annotation on ax2 with high zorder so it sits in front of both lines.
ax2.text(DEFAULT_T + 0.005, 0.05, "Default (0.232)",
         color="#b55", ha="left", va="bottom", fontsize=9,
         transform=ax.transData, zorder=10)
ax2.set_ylabel("Cumulative # Hits", fontsize=10, color="#888")
ax2.tick_params(axis="y", colors="#888")
ax2.spines["top"].set_visible(False)

ax.set_xlim(0, scope["distance"].max() * 1.02)
ax.set_ylim(-0.02, 1.08)
ax.set_xlabel("Distance Threshold")
ax.set_ylabel("Precision")
ax.spines["top"].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_ITER / "threshold_vs_precision.png")
plt.show()
